# Tsetlin bake-off on a free Colab GPU

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
Stay on the tab. Run is ~10–15 min.

**For real data:** add a Colab secret (🔑 sidebar) named `KAGGLE_API_TOKEN`
= your `KGAT_...` token. Cell 2 then downloads the *Beat The Bookie* dataset
(433 real EPL matches). Without it, cell 2 falls back to synthetic data.

Colab is on Python 3.13; `tmu` only ships wheels to 3.12, so cell 3 uses a
throwaway Python 3.11. Everything runs as a subprocess.

In [ ]:
# 1. GPU + code + baseline deps
!nvidia-smi -L || echo 'NO GPU — Runtime > Change runtime type > T4 GPU'
import os
if not os.path.isdir('tsetlin-market-lab'):
    !git clone --depth 1 https://github.com/naibwedi/tsetlin-market-lab.git
os.chdir('/content/tsetlin-market-lab')
!git pull -q
!pip -q install pandas pyarrow pyyaml python-dotenv scikit-learn xgboost lightgbm kaggle
print('cwd', os.getcwd())

In [ ]:
# 2. Data -> features -> the 7 baselines
import glob
try:
    from google.colab import userdata
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
except Exception:
    pass

if not glob.glob('data/features/X.parquet'):
    if os.environ.get('KAGGLE_API_TOKEN', '').startswith('KGAT_'):
        print('Downloading Beat The Bookie (real data)...')
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi(); api.authenticate()
        api.dataset_download_files('austro/beat-the-bookie-worldwide-football-dataset',
                                   path='data/raw/btb', unzip=True, quiet=True)
        !python -m src.ingest.btb --leagues "England: Premier League" --min-books 6
    else:
        print('No KAGGLE_API_TOKEN secret -> synthetic data')
        !python -m src.ingest.make_synthetic --n-matches 80
    !python -m src.panel.build_panel --config config/features.yaml
    !python -m src.features.booleanize --config config/features.yaml
!python -m src.models.bakeoff --config config/bakeoff.ci.yaml
print('\n' + open('results/summary.md').read())

In [ ]:
# 3. Tsetlin Machine on the GPU (Python 3.11 venv; tmu has no 3.13 wheel)
import os
if not os.path.isfile('/content/tm311/bin/python'):
    !pip -q install uv
    !UV_VENV_CLEAR=1 uv venv /content/tm311 --python 3.11 --quiet
    !uv pip install -q --python /content/tm311/bin/python \
        'numpy<2' 'scikit-learn==1.5.2' pandas pyarrow pyyaml python-dotenv tmu pycuda
!cd /content/tsetlin-market-lab && git pull -q
!cd /content/tsetlin-market-lab && /content/tm311/bin/python -m scripts.tm_run

In [ ]:
# 4. The result + the rules it learned
print(open('results/tm_result.json').read())
print('\n--- clauses ---')
print(open('results/tm_clauses.txt').read())